# Experiment 11: Building an RNN to Perform Character-Level Language Modeling

**Aim:** Train a recurrent neural network that predicts the *next character* given the previous characters, and then use it to generate new text.

**Dataset:** Tiny Shakespeare (~1.1M characters of Shakespeare plays) - a standard corpus for char-RNNs.

**Model:** `Embedding -> SimpleRNN(256) -> Dense(vocab)`. A GRU variant is trained as well to show how gated cells improve on the vanilla RNN. Both are trained on the task: input = characters `t[0..n-1]`, target = characters `t[1..n]`.

**Metrics:** cross-entropy loss and **perplexity** = exp(loss).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(7); np.random.seed(7)
SEQ_LEN, BATCH, UNITS, EMB = 100, 128, 256, 64
CHARS_USED = 400_000          # use fewer characters for speed; raise to 1_000_000 for better text
EPOCHS = 8
print("TensorFlow", tf.__version__)

## 1. Load the text

In [ ]:
path = keras.utils.get_file("shakespeare.txt", "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
text = open(path, "rb").read().decode("utf-8")[:CHARS_USED]
print("Characters:", len(text)); print(text[:300])

vocab = sorted(set(text)); VOCAB = len(vocab)
char2idx = {c: i for i, c in enumerate(vocab)}; idx2char = np.array(vocab)
print("Vocabulary size:", VOCAB, vocab[:20], "...")
data = np.array([char2idx[c] for c in text], dtype=np.int32)

## 2. Create training sequences
Overlapping windows of `SEQ_LEN+1` characters are cut from the text (stride 50); the first `SEQ_LEN` characters are the input, the last `SEQ_LEN` (shifted by one) are the target. 90% train / 10% validation (split by position).

In [ ]:
def make_sequences(arr, stride=50):
    starts = np.arange(0, len(arr) - SEQ_LEN - 1, stride)
    X = np.stack([arr[s:s+SEQ_LEN] for s in starts]); Y = np.stack([arr[s+1:s+SEQ_LEN+1] for s in starts])
    return X, Y

split = int(len(data) * 0.9)
X_tr, Y_tr = make_sequences(data[:split]); X_va, Y_va = make_sequences(data[split:])
print("Train:", X_tr.shape, " Val:", X_va.shape)
mk = lambda X, Y, sh: tf.data.Dataset.from_tensor_slices((X, Y)).shuffle(len(X)) .batch(BATCH, drop_remainder=True).prefetch(2) if sh else tf.data.Dataset.from_tensor_slices((X, Y)).batch(BATCH, drop_remainder=True)
train_ds, val_ds = mk(X_tr, Y_tr, True), mk(X_va, Y_va, False)

## 3. Define the models

In [ ]:
def build_model(cell="rnn"):
    rnn = layers.SimpleRNN(UNITS, return_sequences=True) if cell == "rnn" else layers.GRU(UNITS, return_sequences=True)
    return keras.Sequential([layers.Input((None,)), layers.Embedding(VOCAB, EMB), rnn, layers.Dense(VOCAB)], name=f"char_{cell}")

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
models, histories = {}, {}
for cell in ["rnn", "gru"]:
    m = build_model(cell); m.compile(optimizer=keras.optimizers.Adam(2e-3, clipnorm=1.0), loss=loss_fn)
    models[cell] = m
models["rnn"].summary()

## 4. Train both models
Gradient clipping (`clipnorm=1.0`) is used because vanilla RNNs are prone to exploding gradients.

In [ ]:
for cell, m in models.items():
    t0 = time.time(); print(f"\n=== Training {cell.upper()} ===")
    histories[cell] = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2)
    print(f"time: {time.time()-t0:.0f}s")

## 5. Loss and perplexity

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for cell, h in histories.items():
    ax[0].plot(h.history["val_loss"], label=cell.upper())
    ax[1].plot(np.exp(h.history["val_loss"]), label=cell.upper())
ax[0].set_title("Validation loss"); ax[1].set_title("Validation perplexity"); ax[0].legend(); ax[1].legend()
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()
for cell, h in histories.items(): print(f"{cell.upper():4s} final val loss {h.history['val_loss'][-1]:.3f} | perplexity {np.exp(h.history['val_loss'][-1]):.2f}")

## 6. Text generation
We feed a seed string, take the logits for the last character, divide by a **temperature**, sample the next character, append it, and repeat. Low temperature = safe/repetitive, high temperature = creative/noisy.

In [ ]:
def generate(model, seed, n=400, temperature=0.8):
    ids = [char2idx.get(c, 0) for c in seed]
    for _ in range(n):
        logits = model(np.array([ids[-SEQ_LEN:]]), training=False)[0, -1] / temperature
        ids.append(int(tf.random.categorical(logits[None], 1)[0, 0]))
    return "".join(idx2char[i] for i in ids)

for cell in ["rnn", "gru"]:
    for T in (0.5, 1.0):
        print(f"\n######## {cell.upper()} | temperature={T} ########")
        print(generate(models[cell], "ROMEO:\n", 350, T))

## Conclusion
A character-level RNN learned spelling, punctuation, line layout and speaker-name formatting of Shakespeare purely from next-character prediction. The GRU reaches a clearly lower perplexity than the vanilla RNN because its gates mitigate vanishing gradients and keep longer-range context. Sampling temperature trades off coherence vs diversity.